# Phase 2 — Step 2.1: Data Exploration

Run after Phase 1 has produced X.npy / y.npy / label_map.json.

## Cell 1 — Imports

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"]     = 100

print("Libraries ready.")
print(f"  numpy      : {np.__version__}")
print(f"  pandas     : {pd.__version__}")
print(f"  matplotlib : {plt.matplotlib.__version__}")
print(f"  seaborn    : {sns.__version__}")

## Cell 2 — Mount Drive and load data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/nsl-data'

X = np.load(os.path.join(DATA_DIR, 'X.npy'))
y = np.load(os.path.join(DATA_DIR, 'y.npy'))

with open(os.path.join(DATA_DIR, 'label_map.json')) as f:
    label_map = json.load(f)

idx_to_label = {v: k for k, v in label_map.items()}

print(f"X shape  : {X.shape}   dtype: {X.dtype}")
print(f"y shape  : {y.shape}   dtype: {y.dtype}")
print(f"Classes  : {len(label_map)}")
print(f"Labels   : {label_map}")

## Cell 3 — Sanity checks

In [ ]:
print("=" * 60)
print("DATASET SANITY CHECKS")
print("=" * 60)

assert X.ndim == 3,                        f"X must be 3-D, got {X.ndim}-D"
assert X.shape[1] == 30,                   f"X must have 30 frames, got {X.shape[1]}"
assert X.shape[2] == 1662,                 f"X must have 1662 features, got {X.shape[2]}"
assert X.dtype == np.float32,              f"X must be float32, got {X.dtype}"
assert y.ndim == 1,                        f"y must be 1-D, got {y.ndim}-D"
assert y.shape[0] == X.shape[0],           f"X and y lengths differ"
assert set(np.unique(y)).issubset(set(range(len(label_map)))), \
    f"y contains unknown labels"

print(f"  ok  X is 3-D float32 of shape {X.shape}")
print(f"  ok  y is 1-D int of length {y.shape[0]}")
print(f"  ok  All labels are within [0, {len(label_map)-1}]")

finite_mask = np.isfinite(X).all(axis=(1, 2))
n_bad = int((~finite_mask).sum())
if n_bad == 0:
    print(f"  ok  No NaN/Inf values in any sample")
else:
    print(f"  warn  {n_bad} samples contain NaN/Inf -- will need to drop")

x_min, x_max = X[:, :, 0::3].min(), X[:, :, 0::3].max()
y_min, y_max = X[:, :, 1::3].min(), X[:, :, 1::3].max()
print(f"  ok  x range : [{x_min:.3f}, {x_max:.3f}]  (expect ~[0, 1])")
print(f"  ok  y range : [{y_min:.3f}, {y_max:.3f}]  (expect ~[0, 1])")
print(f"  ok  Total samples: {X.shape[0]}")
print(f"  ok  Total classes: {len(label_map)}")
print("=" * 60)

## Cell 4 — Class distribution

In [ ]:
counts = pd.Series(y).value_counts().sort_index()
counts.index = [idx_to_label[i] for i in counts.index]

fig, ax = plt.subplots(figsize=(10, max(4, len(counts) * 0.4)))
bars = ax.barh(counts.index, counts.values,
               color=sns.color_palette("viridis", len(counts)))
ax.set_xlabel("Number of sequences")
ax.set_title(f"Class distribution  (N = {len(y)} sequences across {len(label_map)} signs)")

for bar in bars:
    width = bar.get_width()
    ax.text(width + max(counts.values) * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{int(width)}",
            ha="left", va="center", fontsize=10)

plt.tight_layout()
plt.show()

print("\nClass counts:")
print(counts.to_string())
print(f"\nMean   : {counts.mean():.1f} samples per class")
print(f"Median : {counts.median():.1f}")
print(f"Min    : {counts.min()}")
print(f"Max    : {counts.max()}")
print(f"Imbalance ratio (max/min) : {counts.max() / max(counts.min(), 1):.2f}")

**What the numbers mean in practice:**
- 20+ samples per class = LSTM can start to learn real patterns
- 10-20 per class = usable but noisy results
- Below 10 per class = mostly memorising noise, results not meaningful

## Cell 5 — Sequence length check

In [ ]:
frame_counts = np.full(X.shape[0], X.shape[1], dtype=int)
print(f"Frame counts across all {len(frame_counts)} samples:")
print(f"  Unique values : {np.unique(frame_counts)}")
print(f"  All equal to SEQUENCE_LENGTH ({X.shape[1]}): "
      f"{(frame_counts == X.shape[1]).all()}")

## Cell 6 — Per-class landmark statistics

In [ ]:
class_means = {}
class_stds  = {}
for label_idx, label_name in idx_to_label.items():
    mask = (y == label_idx)
    class_means[label_name] = X[mask].mean(axis=(0, 1))
    class_stds[label_name]  = X[mask].std(axis=(0, 1))

means_df = pd.DataFrame(class_means)
stds_df  = pd.DataFrame(class_stds)

print("Per-class landmark means (first 5 features shown):")
print(means_df.iloc[:5].round(3))
print(f"\nOverall mean range across classes per feature:")
print(f"  Min    : {means_df.min(axis=1).mean():.4f}")
print(f"  Max    : {means_df.max(axis=1).mean():.4f}")
spread = (means_df.max(axis=1) - means_df.min(axis=1)).mean()
print(f"  Spread : {spread:.4f}")
print("\nInterpretation: spread > 0.05 means classes differ meaningfully.")

## Cell 7 — Pairwise class separability heatmap

In [ ]:
class_names = list(class_means.keys())
n_classes   = len(class_names)

dist_matrix = np.zeros((n_classes, n_classes))
for i, ci in enumerate(class_names):
    for j, cj in enumerate(class_names):
        dist_matrix[i, j] = np.mean(
            np.abs(class_means[ci] - class_means[cj])
        )

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    dist_matrix,
    annot=True, fmt=".3f",
    xticklabels=class_names, yticklabels=class_names,
    cmap="YlOrRd", ax=ax,
    cbar_kws={"label": "Mean |delta| between per-feature means"},
)
ax.set_title("Pairwise class separability\n(higher = more visually distinguishable)")
plt.tight_layout()
plt.show()

## Cell 8 — Missing data analysis

In [ ]:
POSE_END   = 33 * 4
FACE_END   = POSE_END + 468 * 3
LH_END     = FACE_END + 21 * 3
RH_END     = LH_END   + 21 * 3

def count_zero_frames(X_subset, start, end):
    block = X_subset[:, :, start:end]
    return np.all(block == 0.0, axis=2).sum(axis=1)

missing = {
    "pose":       count_zero_frames(X, 0,       POSE_END),
    "face":       count_zero_frames(X, POSE_END, FACE_END),
    "left_hand":  count_zero_frames(X, FACE_END, LH_END),
    "right_hand": count_zero_frames(X, LH_END,   RH_END),
}
missing_df = pd.DataFrame(missing)

fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=missing_df, ax=ax, palette="Set2")
ax.set_ylabel("Zero frames per sample (out of 30)")
ax.set_title("Missing landmark frames per body part\n(higher = MediaPipe failed to detect that part more often)")
plt.tight_layout()
plt.show()

print("\nMean zero frames per sample (out of 30):")
print(missing_df.mean().round(2))
n_bad = (missing_df > 10).any(axis=1).sum()
print(f"\nSamples with > 10 missing frames for any body part:")
print(f"  {n_bad} / {len(X)} samples ({100 * n_bad / len(X):.1f}%)")

## Cell 9 — Landmark trajectory visualisation

In [ ]:
RH_START        = LH_END
FINGERTIP_X     = RH_START + 8 * 3
FINGERTIP_Y     = RH_START + 8 * 3 + 1

fig, ax = plt.subplots(figsize=(9, 7))

colors = sns.color_palette("husl", len(label_map))
for color, (label_idx, label_name) in zip(colors, idx_to_label.items()):
    mask = (y == label_idx)
    sample = X[mask][0]
    xs = sample[:, FINGERTIP_X]
    ys = sample[:, FINGERTIP_Y]

    ax.plot(xs, ys, "o-", color=color, alpha=0.7,
            label=label_name, markersize=4)
    ax.scatter(xs[0],  ys[0],  color=color, s=120,
               marker="o", edgecolor="black", linewidth=1.5, zorder=5)
    ax.scatter(xs[-1], ys[-1], color=color, s=120,
               marker="X", edgecolor="black", linewidth=1.5, zorder=5)

ax.set_xlabel("x  (0 = left edge, 1 = right edge)")
ax.set_ylabel("y  (0 = top, 1 = bottom)")
ax.set_title("Right hand index fingertip trajectory across 30 frames\ncircle = first frame   X = last frame")
ax.invert_yaxis()
ax.legend(loc="best", fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Cell 10 — Final exploration summary

In [ ]:
print("=" * 60)
print("EXPLORATION SUMMARY")
print("=" * 60)
print(f"Dataset size        : {X.shape[0]} sequences")
print(f"Number of classes   : {len(label_map)}")
print(f"Sequence length     : {X.shape[1]} frames  (1 second at 30 fps)")
print(f"Feature dimension   : {X.shape[2]}  (pose + face + hands)")
imbalance = counts.max() / max(counts.min(), 1)
print(f"Class balance ratio : {imbalance:.2f}  (< 3 = balanced, > 5 = needs class weights)")
upper = dist_matrix[np.triu_indices(n_classes, k=1)]
mean_sep = float(upper.mean())
print(f"Mean separability   : {mean_sep:.4f}  (> 0.05 = learnable)")

if counts.min() < 10:
    verdict = "INSUFFICIENT DATA -- need at least 10 sequences per class."
elif imbalance > 5:
    verdict = "IMBALANCED -- need class weights at training time."
elif mean_sep < 0.02:
    verdict = "LOW SEPARABILITY -- signs look too similar."
else:
    verdict = "READY FOR TRAINING -- proceed to 02_model_training.ipynb"

print(f"\nVerdict: {verdict}")
print("=" * 60)